# Backtest Comparativo: Long Straddle Sin/Con Hedge

Notebook para comparar el rendimiento de estrategias de long straddle periodico con y sin delta hedging.

**Configuracion simple:**
- Entradas periodicas (sin filtros de senales avanzadas)
- Salida solo por time stop fraction
- Hedge parametrizable (frecuencia de rebalanceo, costes)

---

## Fundamentos del Delta Hedging

### Objetivo
Un long straddle tiene delta inicial cercano a cero (ATM), pero este cambia cuando el precio spot se aleja del strike. El **delta hedging** mantiene la cartera delta-neutral, eliminando la exposicion direccional y transformando la estrategia en una **apuesta pura sobre volatilidad**.

### Fundamento Teorico
El P&L de una posicion delta-hedged se aproxima mediante:

$$
P\&L_{hedged} \approx \frac{1}{2} \times \Gamma \times S^2 \times \left[\left(\frac{\Delta S}{S}\right)^2 - \sigma_{impl}^2 \times \Delta t\right]
$$

**Interpretacion:**
- Si **volatilidad realizada > volatilidad implicita** → Beneficios
- Si **volatilidad realizada < volatilidad implicita** → Perdidas
- El delta hedging **elimina dependencia direccional** pero introduce **path dependency**

### Hedge a Nivel de Cartera
En este backtest, el hedge se aplica **a nivel agregado** cuando hay multiples straddles abiertos simultaneamente:
- Se calcula el **delta total** de todas las posiciones activas
- Se mantiene una **unica posicion de cobertura** en acciones del subyacente
- **Ventajas:** Menor coste de transaccion, refleja practica institucional


## 1. Importaciones y Configuracion del Entorno

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Agregar el directorio scripts al path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
# Esto hace que el notebook guarde una copia en PNG dentro del archivo
# En lugar de usar pio.renderers.default = "notebook_connected+png"
# Prueba a usar solo el modo imagen para ver si así "despierta"

# Modulos del proyecto
from data_loader import get_spy_history, get_vix_history, load_treasury_curve
from strategy import generate_entry_dates
from backtest import run_backtest, prepare_market_data
from signals import EntryConfig, ExitConfig
from delta_hedge import HedgeConfig
from straddle import calculate_straddle_greeks
from rates import get_risk_free_rate

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 200)

print("Modulos cargados correctamente.")

Modulos cargados correctamente.


## 2. Carga de Datos de Mercado

In [2]:
# Periodo de backtest
START_DATE = '2019-01-01'
END_DATE = '2024-01-01'

print(f"Periodo: {START_DATE} a {END_DATE}")
print("Cargando datos...")

Periodo: 2019-01-01 a 2024-01-01
Cargando datos...


In [3]:
# Cargar SPY
spy_data = get_spy_history(START_DATE, END_DATE)
print(f"SPY: {len(spy_data)} dias, {spy_data.index[0]} a {spy_data.index[-1]}")

# Cargar VIX
vix_data = get_vix_history(START_DATE, END_DATE)
print(f"VIX: {len(vix_data)} dias, min={vix_data.min():.2%}, max={vix_data.max():.2%}")

# Cargar Treasury
treasury_data = load_treasury_curve(START_DATE, END_DATE)
print(f"Treasury: {len(treasury_data)} dias, columnas: {list(treasury_data.columns)}")

SPY: 1258 dias, 2019-01-02 a 2023-12-29
VIX: 1258 dias, min=11.54%, max=82.69%
Treasury: 1304 dias, columnas: [30, 90, 180, 365]


In [4]:
# Consolidar datos de mercado
market_data = prepare_market_data(spy_data, vix_data, treasury_data)

print(f"\nMarket Data consolidado:")
print(f"  Shape: {market_data.shape}")
print(f"  Rango: {market_data.index[0]} a {market_data.index[-1]}")
market_data.head()


Market Data consolidado:
  Shape: (1258, 7)
  Rango: 2019-01-02 a 2023-12-29


,close_spy,q_yield,close_vix,30,90,180,365
Date,,,,,,,
2019-01-02,250.179993,0.020389,0.2322,0.0240,0.0242,0.0251,0.0260
2019-01-03,244.210007,0.020888,0.2545,0.0242,0.0241,0.0247,0.0250
2019-01-04,252.389999,0.020211,0.2138,0.0240,0.0242,0.0251,0.0257
2019-01-07,254.380005,0.020053,0.2140,0.0242,0.0245,0.0254,0.0258
2019-01-08,256.769989,0.019866,0.2047,0.0240,0.0246,0.0254,0.0260


## 3. Configuracion de Parametros

Parametros centralizados para el backtest:
- **Frecuencia de entrada:** monthly, weekly, biweekly
- **Tenor:** dias hasta vencimiento de los straddles
- **Time stop fraction:** fraccion del tiempo para salida anticipada
- **Hedge:** frecuencia de rebalanceo y costes de transaccion

In [5]:
# =============================================================================
# PARAMETROS DE BACKTEST
# =============================================================================

# --- Frecuencia y Tenor ---
ENTRY_FREQUENCY = 'monthly'  # 'weekly', 'monthly', 'biweekly'
TENOR_DAYS = 30

# --- Configuracion de Entrada (modo simple: sin filtros) ---
entry_config = EntryConfig(use_filters=False)

# --- Configuracion de Salida (solo time stop) ---
exit_config = ExitConfig(
    use_exits=True,
    take_profit_pct=99.0,      # Deshabilitado (valor muy alto)
    stop_loss_pct=99.0,        # Deshabilitado (valor muy alto)
    time_stop_fraction=0.50,   # PARAMETRO PRINCIPAL: salir al 50% del tiempo
    time_stop_min_return=0.0,  # Salir siempre al time stop (sin minimo de return)
    ivp_exit_threshold=999.0   # Deshabilitado
)

# --- Configuracion de Hedge ---
hedge_config = HedgeConfig(
    rebalance_freq='daily',     # 'daily', 'weekly', 'threshold'
    delta_threshold=0.10,       # Solo para modo 'threshold'
    include_costs=True,
    cost_per_share=0.01,
    multiplier=100
)

# --- Mostrar configuracion ---
print("CONFIGURACION DEL BACKTEST")
print("=" * 50)
print(f"Frecuencia de entrada: {ENTRY_FREQUENCY}")
print(f"Tenor: {TENOR_DAYS} dias")
print(f"\nEntrada: use_filters={entry_config.use_filters}")
print(f"\nSalida:")
print(f"  time_stop_fraction: {exit_config.time_stop_fraction}")
print(f"  time_stop_min_return: {exit_config.time_stop_min_return}")
print(f"\nHedge:")
print(f"  rebalance_freq: {hedge_config.rebalance_freq}")
print(f"  include_costs: {hedge_config.include_costs}")
print(f"  cost_per_share: ${hedge_config.cost_per_share}")

CONFIGURACION DEL BACKTEST
Frecuencia de entrada: monthly
Tenor: 30 dias

Entrada: use_filters=False

Salida:
  time_stop_fraction: 0.5
  time_stop_min_return: 0.0

Hedge:
  rebalance_freq: daily
  include_costs: True
  cost_per_share: $0.01


## 4. Generacion de Fechas de Entrada

In [6]:
# Generar fechas de entrada segun la frecuencia configurada
entry_dates = generate_entry_dates(
    market_data.index[0],
    market_data.index[-1],
    market_data.index,
    ENTRY_FREQUENCY
)

print(f"Fechas de entrada generadas: {len(entry_dates)} trades potenciales")
print(f"Primera entrada: {entry_dates[0]}")
print(f"Ultima entrada: {entry_dates[-1]}")
print(f"\nPrimeras 5 fechas: {entry_dates[:5]}")

Fechas de entrada generadas: 60 trades potenciales
Primera entrada: 2019-01-02
Ultima entrada: 2023-12-01

Primeras 5 fechas: ['2019-01-02', '2019-02-01', '2019-03-01', '2019-04-01', '2019-05-01']


## 5. Backtest SIN Hedge

Ejecuta el backtest de long straddle sin delta hedging.

In [7]:
print("Ejecutando backtest SIN hedge...")

results_no_hedge = run_backtest(
    market_data=market_data,
    entry_dates=entry_dates,
    tenor_days=TENOR_DAYS,
    entry_config=entry_config,
    exit_config=exit_config,
    hedge_config=None  # Sin hedge
)

print("Backtest SIN hedge completado.")
print(f"  Trades ejecutados: {len(results_no_hedge.trades)}")
print(f"  P&L Total: ${results_no_hedge.summary.get('total_pnl', 0):.2f}")

Ejecutando backtest SIN hedge...
Backtest SIN hedge completado.
  Trades ejecutados: 60
  P&L Total: $515.86


In [8]:
# Resumen detallado - Sin Hedge
print("RESUMEN SIN HEDGE")
print("=" * 50)
for key, value in results_no_hedge.summary.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:>12.2f}")
    elif isinstance(value, (int, str)):
        print(f"  {key:25s}: {str(value):>12}")
    else:
        print(f"  {key:25s}: {str(value)}")

RESUMEN SIN HEDGE
  total_pnl_straddle       :      -110.57
  num_trades               :           60
  avg_pnl_per_trade        :        -1.84
  win_rate                 :         0.10
  avg_win                  :        14.68
  avg_loss                 :        -3.68
  best_trade               :        39.26
  worst_trade              :       -13.58
  avg_holding_days         :        17.45
  open_positions           :            0
  exit_reasons             : {'expiry': 6, 'take_profit': 0, 'stop_loss': 0, 'time_stop': 54, 'vol_exit': 0}
  exit_config              : {'take_profit_pct': 99.0, 'stop_loss_pct': 99.0, 'time_stop_fraction': 0.5, 'ivp_exit_threshold': 999.0}
  total_pnl                :       515.86
  max_drawdown             :      -485.29
  profit_factor            :         0.44


## 6. Backtest CON Hedge

Ejecuta el backtest de long straddle con delta hedging activo.

El delta hedging discreto (diario, semanal, etc.) introduce path dependency en los resultados. La frecuencia óptima de rebalanceo depende del régimen de mercado y no puede determinarse ex-ante. En la práctica, el hedge diario minimiza el tracking error respecto al hedge continuo teórico, pero puede incurrir en mayores costes de transacción y "gamma bleeding" en mercados trending.

---

## Arquitectura del Módulo `delta_hedge.py`

Este módulo contiene la implementación completa del sistema de delta hedging mediante tres clases de datos y funciones auxiliares.

### Clase `HedgeConfig`

Dataclass que encapsula todos los parámetros configurables del delta hedge.

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `rebalance_freq` | str | Frecuencia de rebalanceo: `'daily'`, `'weekly'` o `'threshold'` |
| `delta_threshold` | float | Umbral de delta para modo `'threshold'`. Se rebalancea cuando \|Δ_cartera\| > umbral (default: 0.10) |
| `include_costs` | bool | Si se deben incluir costes de transacción (default: False) |
| `cost_per_share` | float | Coste por acción (comisión + slippage). Solo si `include_costs=True` (default: 0.01) |
| `multiplier` | int | Multiplicador del contrato. Para SPY = 100 (cada contrato controla 100 acciones) |

### Clase `HedgeState`

Mantiene el estado mutable del hedge durante la simulación. Permite múltiples simulaciones con la misma configuración pero estados independientes.

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `shares_held` | float | Número de acciones en cartera. (+) largo, (-) corto |
| `cumulative_pnl` | float | P&L acumulado del hedge desde inicio |
| `cumulative_costs` | float | Costes de transacción acumulados |
| `last_rebalance_date` | str | Fecha del último rebalanceo |

### Clase `HedgeTrade`

Registra los detalles de cada operación de rebalanceo. Permite análisis detallado de la actividad del hedge.

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `date` | str | Fecha del rebalanceo |
| `delta_portfolio` | float | Delta agregado de la cartera de opciones |
| `shares_before` | float | Acciones antes del rebalanceo |
| `shares_after` | float | Acciones después del rebalanceo |
| `shares_traded` | float | Acciones negociadas (`shares_after - shares_before`) |
| `spot_price` | float | Precio del subyacente al rebalancear |
| `trade_cost` | float | Coste de transacción de la operación |

In [9]:
print("Ejecutando backtest CON hedge...")

results_with_hedge = run_backtest(
    market_data=market_data,
    entry_dates=entry_dates,
    tenor_days=TENOR_DAYS,
    entry_config=entry_config,
    exit_config=exit_config,
    hedge_config=hedge_config  # Con hedge
)

print("Backtest CON hedge completado.")
print(f"  Trades ejecutados: {len(results_with_hedge.trades)}")
print(f"  P&L Total Straddle: ${results_with_hedge.summary.get('total_pnl', 0):.2f}")
print(f"  P&L Total Hedge: ${results_with_hedge.summary.get('total_pnl_hedge', 0):.2f}")

Ejecutando backtest CON hedge...
Backtest CON hedge completado.
  Trades ejecutados: 60
  P&L Total Straddle: $619.97
  P&L Total Hedge: $0.00


In [10]:
# Resumen detallado - Con Hedge
print("RESUMEN CON HEDGE")
print("=" * 50)
for key, value in results_with_hedge.summary.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:>12.2f}")
    elif isinstance(value, (int, str)):
        print(f"  {key:25s}: {str(value):>12}")
    else:
        print(f"  {key:25s}: {str(value)}")

RESUMEN CON HEDGE
  total_pnl_straddle       :      -110.57
  num_trades               :           60
  avg_pnl_per_trade        :        -1.84
  win_rate                 :         0.10
  avg_win                  :        14.68
  avg_loss                 :        -3.68
  best_trade               :        39.26
  worst_trade              :       -13.58
  avg_holding_days         :        17.45
  open_positions           :            0
  exit_reasons             : {'expiry': 6, 'take_profit': 0, 'stop_loss': 0, 'time_stop': 54, 'vol_exit': 0}
  exit_config              : {'take_profit_pct': 99.0, 'stop_loss_pct': 99.0, 'time_stop_fraction': 0.5, 'ivp_exit_threshold': 999.0}
  total_pnl_hedge          :       104.12
  hedge_num_rebalances     :          779
  hedge_total_costs        :       100.81
  hedge_config             :        daily
  total_pnl                :       619.97
  max_drawdown             :     -5450.09
  profit_factor            :         0.44


## 7. Analisis de Informacion del Hedge

Visualizacion detallada de las operaciones de hedge:
- Tabla de rebalanceos
- Evolucion de acciones en cartera
- P&L del hedge vs P&L del straddle

In [11]:
# Informacion general del hedge
print("INFORMACION DEL HEDGE")
print("=" * 50)
print(f"Rebalanceos totales: {results_with_hedge.summary.get('hedge_num_rebalances', 0)}")
print(f"Costes totales: ${results_with_hedge.summary.get('hedge_total_costs', 0):.2f}")
print(f"P&L Hedge: ${results_with_hedge.summary.get('total_pnl_hedge', 0):.2f}")
print(f"P&L Straddle: ${results_with_hedge.summary.get('total_pnl', 0):.2f}")

# P&L combinado
pnl_combined = results_with_hedge.summary.get('total_pnl', 0) + results_with_hedge.summary.get('total_pnl_hedge', 0)
print(f"\nP&L Combinado (Straddle + Hedge): ${pnl_combined:.2f}")

INFORMACION DEL HEDGE
Rebalanceos totales: 0
Costes totales: $100.81
P&L Hedge: $0.00
P&L Straddle: $619.97

P&L Combinado (Straddle + Hedge): $619.97


In [12]:
# Tabla de rebalanceos (primeros 20)
if hasattr(results_with_hedge, 'hedge_trades') and results_with_hedge.hedge_trades:
    hedge_trades_df = pd.DataFrame([
        {
            'date': t.date,
            'delta_portfolio': t.delta_portfolio,
            'shares_before': t.shares_before,
            'shares_after': t.shares_after,
            'shares_traded': t.shares_traded,
            'spot_price': t.spot_price,
            'trade_cost': t.trade_cost
        }
        for t in results_with_hedge.hedge_trades
    ])
    
    print(f"\nPrimeros 20 rebalanceos (de {len(hedge_trades_df)} totales):")
    print(hedge_trades_df.head(20).to_string(index=False))
else:
    print("No hay datos de hedge trades disponibles.")


Primeros 20 rebalanceos (de 779 totales):
      date  delta_portfolio  shares_before  shares_after  shares_traded  spot_price  trade_cost
2019-01-02         0.038661       0.000000     -3.866063      -3.866063  250.179993    0.038661
2019-01-03        -0.225596      -3.866063     22.559587      26.425649  244.210007    0.264256
2019-01-04         0.154518      22.559587    -15.451794     -38.011380  252.389999    0.380114
2019-01-07         0.268175     -15.451794    -26.817469     -11.365675  254.380005    0.113657
2019-01-08         0.410589     -26.817469    -41.058916     -14.241448  256.769989    0.142414
2019-01-09         0.487630     -41.058916    -48.763047      -7.704131  257.970001    0.077041
2019-01-10         0.551296     -48.763047    -55.129648      -6.366600  258.880005    0.063666
2019-01-11         0.596421     -55.129648    -59.642113      -4.512465  258.980011    0.045125
2019-01-14         0.524934     -59.642113    -52.493371       7.148742  257.399994    0.0714

In [13]:
# Grafico: Evolucion de Shares en Cartera
if hasattr(results_with_hedge, 'hedge_trades') and results_with_hedge.hedge_trades:
    fig = go.Figure()
    
    dates = [t.date for t in results_with_hedge.hedge_trades]
    shares = [t.shares_after for t in results_with_hedge.hedge_trades]
    
    fig.add_trace(go.Scatter(
        x=dates,
        y=shares,
        mode='lines',
        name='Shares en Cartera',
        line=dict(color='blue')
    ))
    
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    
    fig.update_layout(
        title='Evolucion de Acciones en Cartera (Delta Hedge)',
        xaxis_title='Fecha',
        yaxis_title='Numero de Acciones',
        height=400
    )
    
    fig.show()

In [ ]:
# Grafico: P&L Acumulado del Hedge vs Straddle
fig = make_subplots(rows=1, cols=1)

# P&L acumulado del straddle (con hedge)
pnl_straddle = results_with_hedge.cumulative_pnl_straddle
fig.add_trace(go.Scatter(
    x=pnl_straddle.index,
    y=pnl_straddle.values,
    mode='lines',
    name='P&L Straddle',
    line=dict(color='blue')
))

# P&L acumulado del hedge
if results_with_hedge.cumulative_pnl_hedge is not None and len(results_with_hedge.cumulative_pnl_hedge) > 0:
    pnl_hedge = results_with_hedge.cumulative_pnl_hedge
    fig.add_trace(go.Scatter(
        x=pnl_hedge.index,
        y=pnl_hedge.values,
        mode='lines',
        name='P&L Hedge',
        line=dict(color='orange')
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray')

fig.update_layout(
    title='Descomposicion del P&L: Straddle vs Hedge',
    xaxis_title='Fecha',
    yaxis_title='P&L Acumulado ($)',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

## 8. Comparacion de Resultados

Comparacion directa entre la estrategia sin hedge y con hedge:
- Tabla de metricas
- Grafico de P&L acumulado
- Grafico de drawdown

In [ ]:
# Tabla comparativa de metricas
print("COMPARACION DE METRICAS")
print("=" * 70)
print(f"{'Metrica':30s} {'Sin Hedge':>18s} {'Con Hedge':>18s}")
print("-" * 70)

# Metricas a comparar
metricas = [
    ('total_pnl', 'P&L Total Straddle'),
    ('total_pnl_hedge', 'P&L Hedge'),
    ('num_trades', 'Numero de Trades'),
    ('win_rate', 'Win Rate (%)'),
    ('avg_pnl_per_trade', 'P&L Promedio/Trade'),
    ('max_drawdown', 'Max Drawdown'),
    ('profit_factor', 'Profit Factor'),
]

for key, label in metricas:
    val_no_hedge = results_no_hedge.summary.get(key, 0)
    val_hedge = results_with_hedge.summary.get(key, 0)
    
    if isinstance(val_no_hedge, float):
        print(f"{label:30s} {val_no_hedge:>18.2f} {val_hedge:>18.2f}")
    else:
        print(f"{label:30s} {str(val_no_hedge):>18s} {str(val_hedge):>18s}")

# P&L Total combinado (solo para hedge)
pnl_no_hedge = results_no_hedge.summary.get('total_pnl', 0)
pnl_hedge_total = results_with_hedge.summary.get('total_pnl', 0) + results_with_hedge.summary.get('total_pnl_hedge', 0)
print("-" * 70)
print(f"{'P&L TOTAL COMBINADO':30s} {pnl_no_hedge:>18.2f} {pnl_hedge_total:>18.2f}")

COMPARACION DE METRICAS
Metrica                                 Sin Hedge          Con Hedge
----------------------------------------------------------------------
P&L Total Straddle                         515.86             619.97
P&L Hedge                                       0                  0
Numero de Trades                               60                 60
Win Rate (%)                                 0.10               0.10
P&L Promedio/Trade                          -1.84              -1.84
Max Drawdown                              -485.29           -5450.09
Profit Factor                                0.44               0.44
----------------------------------------------------------------------
P&L TOTAL COMBINADO                        515.86             619.97


In [ ]:
def calculate_open_positions_timeseries(results, market_data):
    """
    Calcula el numero de posiciones abiertas en cada dia de trading.

    Parameters
    ----------
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado con indice datetime

    Returns
    -------
    pd.Series
        Numero de posiciones abiertas para cada fecha
    """
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)

    counts = []
    dates = []

    for date in market_data_copy.index:
        date_str = str(date.date())
        count = sum(
            1 for trade in results.trades
            if trade.entry_date <= date_str <= trade.expiry_date
        )
        counts.append(count)
        dates.append(date)

    return pd.Series(counts, index=dates, name="Open Positions")

In [ ]:
# Calcular posiciones abiertas para ambas estrategias
open_pos_no_hedge = calculate_open_positions_timeseries(results_no_hedge, market_data)
open_pos_with_hedge = calculate_open_positions_timeseries(results_with_hedge, market_data)

# Crear figura con eje secundario
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Traces de P&L en eje izquierdo
pnl_no_hedge_series = results_no_hedge.cumulative_pnl_straddle
fig.add_trace(
    go.Scatter(
        x=pnl_no_hedge_series.index,
        y=pnl_no_hedge_series.values,
        mode='lines',
        name='Sin Hedge',
        line=dict(color='blue', width=2)
    ),
    secondary_y=False
)

# Con hedge
if results_with_hedge.cumulative_pnl_total is not None and len(results_with_hedge.cumulative_pnl_total) > 0:
    pnl_hedge_series = results_with_hedge.cumulative_pnl_total
else:
    pnl_hedge_series = results_with_hedge.cumulative_pnl_straddle

fig.add_trace(
    go.Scatter(
        x=pnl_hedge_series.index,
        y=pnl_hedge_series.values,
        mode='lines',
        name='Con Hedge',
        line=dict(color='green', width=2)
    ),
    secondary_y=False
)

# Posiciones abiertas en eje derecho
fig.add_trace(
    go.Scatter(
        x=open_pos_no_hedge.index,
        y=open_pos_no_hedge.values,
        mode='lines',
        name='Posiciones Abiertas',
        line=dict(color='purple', width=1.5, dash='dot'),
        #fill='tozeroy',
        #fillcolor='rgba(128, 0, 128, 0.1)'
    ),
    secondary_y=True
)

fig.add_hline(y=0, line_dash='dash', line_color='gray', secondary_y=False)

# Actualizar ejes
fig.update_yaxes(title_text="P&L Acumulado ($)", secondary_y=False)
fig.update_yaxes(title_text="Numero de Posiciones Abiertas", secondary_y=True)

fig.update_layout(
    title='P&L Acumulado: Sin Hedge vs Con Hedge (con Posiciones Abiertas)',
    xaxis_title='Fecha',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [ ]:
# Grafico: Drawdown Comparativo
def calculate_drawdown(pnl_series):
    """Calcula el drawdown de una serie de P&L acumulado."""
    cummax = pnl_series.cummax()
    drawdown = pnl_series - cummax
    return drawdown

fig = go.Figure()

# Drawdown sin hedge
dd_no_hedge = calculate_drawdown(pnl_no_hedge_series)
fig.add_trace(go.Scatter(
    x=dd_no_hedge.index,
    y=dd_no_hedge.values,
    mode='lines',
    name='Sin Hedge',
    fill='tozeroy',
    line=dict(color='blue')
))

# Drawdown con hedge
dd_hedge = calculate_drawdown(pnl_hedge_series)
fig.add_trace(go.Scatter(
    x=dd_hedge.index,
    y=dd_hedge.values,
    mode='lines',
    name='Con Hedge',
    fill='tozeroy',
    line=dict(color='green')
))

fig.update_layout(
    title='Drawdown Comparativo',
    xaxis_title='Fecha',
    yaxis_title='Drawdown ($)',
    height=400,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

## 9. Evolucion de Griegas del Portfolio

Analisis de las griegas agregadas del portfolio:
- Griegas sin hedge
- Griegas con hedge (delta neto deberia estar cercano a 0)
- Comparacion de delta entre ambas estrategias

In [ ]:
def calculate_portfolio_greeks_timeseries(results, market_data):
    """
    Calcula las griegas agregadas del portfolio para cada dia de trading.
    
    Suma las griegas de todas las posiciones activas en cada fecha.
    
    Parameters
    ----------
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado consolidados
        
    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: delta, gamma, vega, theta, rho, num_positions
    """
    # Asegurar que el indice este en formato datetime
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    # Inicializar diccionario para guardar griegas diarias
    daily_greeks = {
        'delta': [],
        'gamma': [],
        'vega': [],
        'theta': [],
        'rho': [],
        'num_positions': [],
        'date': []
    }
    
    # Para cada dia de trading
    for date in market_data_copy.index:
        date_str = str(date.date())
        
        # Encontrar todas las posiciones activas en esta fecha
        active_positions = [
            trade for trade in results.trades
            if trade.entry_date <= date_str <= trade.expiry_date
        ]
        
        # Sumar griegas de todas las posiciones activas
        total_delta = 0.0
        total_gamma = 0.0
        total_vega = 0.0
        total_theta = 0.0
        total_rho = 0.0
        
        if active_positions:
            row = market_data_copy.loc[date]
            S = row['close_spy']
            sigma = row['close_vix']
            q = row['q_yield']
            
            for trade in active_positions:
                # Calcular tiempo hasta vencimiento
                T = trade.time_to_expiry(date_str)
                
                if T > 0:
                    # Obtener tasa risk-free
                    days_remaining = trade.days_to_expiry(date_str)
                    curve_row = {
                        30: row[30],
                        90: row[90],
                        180: row[180],
                        365: row[365]
                    }
                    r = get_risk_free_rate(days_remaining, curve_row)
                    
                    # Calcular griegas del straddle
                    greeks = calculate_straddle_greeks(S, trade.strike, T, r, q, sigma)
                    
                    total_delta += greeks['delta']
                    total_gamma += greeks['gamma']
                    total_vega += greeks['vega']
                    total_theta += greeks['theta']
                    total_rho += greeks['rho']
        
        # Guardar valores del dia
        daily_greeks['date'].append(date)
        daily_greeks['delta'].append(total_delta)
        daily_greeks['gamma'].append(total_gamma)
        daily_greeks['vega'].append(total_vega)
        daily_greeks['theta'].append(total_theta)
        daily_greeks['rho'].append(total_rho)
        daily_greeks['num_positions'].append(len(active_positions))
    
    # Crear DataFrame
    greeks_df = pd.DataFrame(daily_greeks)
    greeks_df.set_index('date', inplace=True)
    
    return greeks_df

In [ ]:
# Calcular griegas para ambos backtests
print("Calculando griegas del portfolio SIN hedge...")
greeks_no_hedge = calculate_portfolio_greeks_timeseries(results_no_hedge, market_data)

print("Calculando griegas del portfolio CON hedge...")
greeks_with_hedge = calculate_portfolio_greeks_timeseries(results_with_hedge, market_data)

print(f"\nGriegas calculadas para {len(greeks_no_hedge)} dias de trading.")

Calculando griegas del portfolio SIN hedge...
Calculando griegas del portfolio CON hedge...

Griegas calculadas para 1258 dias de trading.


In [ ]:
# Estadisticas de griegas - Sin Hedge
print("ESTADISTICAS DE GRIEGAS - SIN HEDGE")
print("=" * 50)
print(greeks_no_hedge[['delta', 'gamma', 'vega', 'theta']].describe())

ESTADISTICAS DE GRIEGAS - SIN HEDGE
             delta        gamma         vega        theta
count  1258.000000  1258.000000  1258.000000  1258.000000
mean      0.169327     0.039797     0.473058    -0.306429
std       0.545085     0.027038     0.294536     0.218118
min      -1.182312     0.000000     0.000000    -2.221591
25%      -0.140752     0.025658     0.234353    -0.401145
50%       0.175171     0.036257     0.493471    -0.290081
75%       0.598399     0.050018     0.703411    -0.168608
max       1.164228     0.246691     1.118910     0.047302


In [ ]:
# Grafico: Evolucion de Griegas - Sin Hedge
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Delta', 'Gamma', 'Vega', 'Theta')
)

fig.add_trace(
    go.Scatter(x=greeks_no_hedge.index, y=greeks_no_hedge['delta'], 
               mode='lines', name='Delta', line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=greeks_no_hedge.index, y=greeks_no_hedge['gamma'],
               mode='lines', name='Gamma', line=dict(color='green')),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=greeks_no_hedge.index, y=greeks_no_hedge['vega'],
               mode='lines', name='Vega', line=dict(color='orange')),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=greeks_no_hedge.index, y=greeks_no_hedge['theta'],
               mode='lines', name='Theta', line=dict(color='red')),
    row=2, col=2
)

fig.update_layout(
    title='Evolucion de Griegas del Portfolio - SIN Hedge',
    height=600,
    showlegend=False
)

fig.show()

In [ ]:
# Grafico: Comparacion de Delta - Sin Hedge vs Delta Neto Con Hedge
# El delta de las opciones es el mismo, pero el delta NETO (opciones + acciones)
# deberia estar cerca de 0 cuando hay hedge.

fig = go.Figure()

# Delta de las opciones (sin hedge = con hedge, son las mismas posiciones)
fig.add_trace(go.Scatter(
    x=greeks_no_hedge.index,
    y=greeks_no_hedge['delta'],
    mode='lines',
    name='Delta Opciones (Sin Hedge)',
    line=dict(color='blue', width=2)
))

# Delta neto (opciones + hedge con acciones)
if hasattr(results_with_hedge, 'hedge_trades') and results_with_hedge.hedge_trades:
    # Crear serie de delta del hedge (acciones)
    hedge_dates = [t.date for t in results_with_hedge.hedge_trades]
    hedge_shares = [t.shares_after for t in results_with_hedge.hedge_trades]
    hedge_series = pd.Series(hedge_shares, index=pd.to_datetime(hedge_dates))
    
    # Delta de acciones = shares / multiplier
    delta_acciones = hedge_series / hedge_config.multiplier
    
    # Alinear con griegas y calcular delta neto
    delta_acciones_aligned = delta_acciones.reindex(greeks_with_hedge.index, method='ffill').fillna(0)
    delta_neto = greeks_with_hedge['delta'] + delta_acciones_aligned
    
    fig.add_trace(go.Scatter(
        x=delta_neto.index,
        y=delta_neto.values,
        mode='lines',
        name='Delta Neto (Con Hedge)',
        line=dict(color='green', width=2)
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray')

fig.update_layout(
    title='Comparacion de Delta: Opciones vs Delta Neto con Hedge',
    xaxis_title='Fecha',
    yaxis_title='Delta',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

El delta de cartera se calcula como la suma de los deltas individuales de cada straddle abierto. Para cada posicion, se invoca calculate_straddle_greeks() y se extrae el delta. Al vencimiento, cuando T=0, el delta se determina por la moneyness: +1 si el spot supera el strike (call ITM), -1 si el spot esta por debajo (put ITM), y 0 si coinciden exactamente.

## 10. Contribucion de Griegas al P&L

Descomposicion del P&L por contribucion de cada griega:
- P&L por Delta: delta * cambio_spot
- P&L por Gamma: 0.5 * gamma * (cambio_spot)^2
- P&L por Vega: vega * cambio_iv
- P&L por Theta: theta * dt

In [ ]:
def decompose_pnl_by_greeks(greeks_df, market_data, multiplier=100):
    """
    Descompone el P&L por contribucion de cada griega.
    
    Aproximacion de Taylor de primer orden:
    dV = delta * dS + 0.5 * gamma * dS^2 + vega * dIV + theta * dt
    
    Parameters
    ----------
    greeks_df : pd.DataFrame
        DataFrame con griegas diarias del portfolio
    market_data : pd.DataFrame
        Datos de mercado consolidados
    multiplier : int
        Multiplicador del contrato
        
    Returns
    -------
    pd.DataFrame
        DataFrame con contribucion diaria de cada griega al P&L
    """
    # Asegurar indices datetime
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    # Alinear indices
    common_dates = greeks_df.index.intersection(market_data_copy.index)
    greeks_aligned = greeks_df.loc[common_dates]
    market_aligned = market_data_copy.loc[common_dates]
    
    # Calcular cambios diarios
    dS = market_aligned['close_spy'].diff()  # Cambio en spot
    dIV = market_aligned['close_vix'].diff()  # Cambio en IV (en decimal)
    dt = 1 / 252  # Fraccion de ano por dia
    
    # Usar griegas del dia anterior para calcular P&L del dia
    delta_prev = greeks_aligned['delta'].shift(1)
    gamma_prev = greeks_aligned['gamma'].shift(1)
    vega_prev = greeks_aligned['vega'].shift(1)
    theta_prev = greeks_aligned['theta'].shift(1)
    
    # Calcular contribuciones (ajustadas por multiplicador)
    pnl_delta = delta_prev * dS * multiplier
    pnl_gamma = 0.5 * gamma_prev * (dS ** 2) * multiplier
    pnl_vega = vega_prev * dIV * 100 * multiplier  # dIV en %, vega por punto de vol
    pnl_theta = theta_prev * dt * multiplier
    
    # Crear DataFrame de resultados
    decomposition = pd.DataFrame({
        'pnl_delta': pnl_delta,
        'pnl_gamma': pnl_gamma,
        'pnl_vega': pnl_vega,
        'pnl_theta': pnl_theta,
        'pnl_total_approx': pnl_delta + pnl_gamma + pnl_vega + pnl_theta
    })
    
    return decomposition

In [ ]:
# Descomponer P&L para ambas estrategias
print("Calculando descomposicion del P&L por griegas...")

pnl_decomp_no_hedge = decompose_pnl_by_greeks(greeks_no_hedge, market_data)
pnl_decomp_with_hedge = decompose_pnl_by_greeks(greeks_with_hedge, market_data)

print("Descomposicion completada.")

Calculando descomposicion del P&L por griegas...
Descomposicion completada.


In [ ]:
# Grafico: Contribucion Acumulada por Griega - SIN Hedge
fig = go.Figure()

# Acumulados
cum_delta = pnl_decomp_no_hedge['pnl_delta'].cumsum()
cum_gamma = pnl_decomp_no_hedge['pnl_gamma'].cumsum()
cum_vega = pnl_decomp_no_hedge['pnl_vega'].cumsum()
cum_theta = pnl_decomp_no_hedge['pnl_theta'].cumsum()

fig.add_trace(go.Scatter(
    x=cum_delta.index, y=cum_delta.values,
    mode='lines', name='Delta',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_gamma.index, y=cum_gamma.values,
    mode='lines', name='Gamma',
    line=dict(color='green', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_vega.index, y=cum_vega.values,
    mode='lines', name='Vega',
    line=dict(color='orange', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_theta.index, y=cum_theta.values,
    mode='lines', name='Theta',
    line=dict(color='red', width=2)
))

# P&L real para comparar
pnl_real = results_no_hedge.cumulative_pnl_straddle
fig.add_trace(go.Scatter(
    x=pnl_real.index, y=pnl_real.values,
    mode='lines', name='P&L Real',
    line=dict(color='black', width=2, dash='dash')
))

fig.add_hline(y=0, line_dash='dot', line_color='gray')

fig.update_layout(
    title='Contribucion Acumulada por Griega al P&L - SIN Hedge',
    xaxis_title='Fecha',
    yaxis_title='P&L Acumulado ($)',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [ ]:
# Grafico: Contribucion Acumulada por Griega - CON Hedge
# Mostramos la contribucion de las opciones + el P&L del hedge con acciones

fig = go.Figure()

# Acumulados de opciones
cum_delta_h = pnl_decomp_with_hedge['pnl_delta'].cumsum()
cum_gamma_h = pnl_decomp_with_hedge['pnl_gamma'].cumsum()
cum_vega_h = pnl_decomp_with_hedge['pnl_vega'].cumsum()
cum_theta_h = pnl_decomp_with_hedge['pnl_theta'].cumsum()

fig.add_trace(go.Scatter(
    x=cum_delta_h.index, y=cum_delta_h.values,
    mode='lines', name='Delta (Opciones)',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_gamma_h.index, y=cum_gamma_h.values,
    mode='lines', name='Gamma',
    line=dict(color='green', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_vega_h.index, y=cum_vega_h.values,
    mode='lines', name='Vega',
    line=dict(color='orange', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_theta_h.index, y=cum_theta_h.values,
    mode='lines', name='Theta',
    line=dict(color='red', width=2)
))

# P&L del hedge (acciones)
if results_with_hedge.cumulative_pnl_hedge is not None and len(results_with_hedge.cumulative_pnl_hedge) > 0:
    pnl_hedge_accum = results_with_hedge.cumulative_pnl_hedge
    fig.add_trace(go.Scatter(
        x=pnl_hedge_accum.index, y=pnl_hedge_accum.values,
        mode='lines', name='P&L Hedge (Acciones)',
        line=dict(color='purple', width=2)
    ))

# P&L total real
if results_with_hedge.cumulative_pnl_total is not None and len(results_with_hedge.cumulative_pnl_total) > 0:
    pnl_total_real = results_with_hedge.cumulative_pnl_total
    fig.add_trace(go.Scatter(
        x=pnl_total_real.index, y=pnl_total_real.values,
        mode='lines', name='P&L Total Real',
        line=dict(color='black', width=2, dash='dash')
    ))

fig.add_hline(y=0, line_dash='dot', line_color='gray')

fig.update_layout(
    title='Contribucion Acumulada por Griega al P&L - CON Hedge',
    xaxis_title='Fecha',
    yaxis_title='P&L Acumulado ($)',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [ ]:
# Tabla Resumen: Contribucion Total de cada Griega
print("CONTRIBUCION TOTAL DE GRIEGAS AL P&L")
print("=" * 70)
print(f"{'Componente':20s} {'Sin Hedge ($)':>20s} {'Con Hedge ($)':>20s}")
print("-" * 70)

componentes = [
    ('pnl_delta', 'Delta'),
    ('pnl_gamma', 'Gamma'),
    ('pnl_vega', 'Vega'),
    ('pnl_theta', 'Theta'),
]

for col, label in componentes:
    total_no_hedge = pnl_decomp_no_hedge[col].sum()
    total_with_hedge = pnl_decomp_with_hedge[col].sum()
    print(f"{label:20s} {total_no_hedge:>20.2f} {total_with_hedge:>20.2f}")

print("-" * 70)

# Suma aproximada vs P&L real
total_approx_no_hedge = pnl_decomp_no_hedge['pnl_total_approx'].sum()
total_approx_with_hedge = pnl_decomp_with_hedge['pnl_total_approx'].sum()

print(f"{'Suma Griegas (Aprox)':20s} {total_approx_no_hedge:>20.2f} {total_approx_with_hedge:>20.2f}")

# P&L real
pnl_real_no_hedge = results_no_hedge.summary.get('total_pnl', 0)
pnl_real_with_hedge = results_with_hedge.summary.get('total_pnl', 0)

print(f"{'P&L Real Straddle':20s} {pnl_real_no_hedge:>20.2f} {pnl_real_with_hedge:>20.2f}")

# Hedge P&L (solo con hedge)
hedge_pnl = results_with_hedge.summary.get('total_pnl_hedge', 0)
print(f"{'P&L Hedge':20s} {'-':>20s} {hedge_pnl:>20.2f}")
print(f"{'P&L Total Combinado':20s} {pnl_real_no_hedge:>20.2f} {pnl_real_with_hedge + hedge_pnl:>20.2f}")

CONTRIBUCION TOTAL DE GRIEGAS AL P&L
Componente                  Sin Hedge ($)        Con Hedge ($)
----------------------------------------------------------------------
Delta                             4371.53              4371.53
Gamma                            38696.75             38696.75
Vega                              -229.75              -229.75
Theta                             -152.97              -152.97
----------------------------------------------------------------------
Suma Griegas (Aprox)             42685.55             42685.55
P&L Real Straddle                  515.86               619.97
P&L Hedge                               -                 0.00
P&L Total Combinado                515.86               619.97


## 11. Verificaciones Finales

In [ ]:
print("VERIFICACIONES FINALES")
print("=" * 60)

# 1. Numero de trades consistente
trades_no_hedge = len(results_no_hedge.trades)
trades_with_hedge = len(results_with_hedge.trades)
trades_ok = trades_no_hedge == trades_with_hedge
print(f"1. Trades sin hedge: {trades_no_hedge}, con hedge: {trades_with_hedge}")
print(f"   Consistente: {'OK' if trades_ok else 'ERROR'}")

# 2. Todos los trades cerrados (excepto posiblemente el ultimo)
open_no_hedge = sum(1 for t in results_no_hedge.trades if t.status == 'open')
open_with_hedge = sum(1 for t in results_with_hedge.trades if t.status == 'open')
print(f"\n2. Trades abiertos: sin hedge={open_no_hedge}, con hedge={open_with_hedge}")
print(f"   OK (puede haber 0-1 al final)")

# 3. Strikes ATM
strikes = [t.strike for t in results_no_hedge.trades]
all_integer = all(s == int(s) for s in strikes)
print(f"\n3. Strikes son enteros (ATM): {'OK' if all_integer else 'ERROR'}")

# 4. Entry prices razonables
entry_prices = [t.entry_price for t in results_no_hedge.trades]
prices_ok = all(3 < p < 60 for p in entry_prices)
print(f"\n4. Entry prices razonables ($3-$60): {'OK' if prices_ok else 'REVISAR'}")
print(f"   Min: ${min(entry_prices):.2f}, Max: ${max(entry_prices):.2f}")

# 5. Hedge trades existen
has_hedge_trades = hasattr(results_with_hedge, 'hedge_trades') and len(results_with_hedge.hedge_trades) > 0
print(f"\n5. Hedge trades registrados: {'OK' if has_hedge_trades else 'ERROR'}")
if has_hedge_trades:
    print(f"   Total rebalanceos: {len(results_with_hedge.hedge_trades)}")

# 6. Griegas calculadas
greeks_ok = len(greeks_no_hedge) > 0 and len(greeks_with_hedge) > 0
print(f"\n6. Griegas calculadas: {'OK' if greeks_ok else 'ERROR'}")
print(f"   Dias con griegas: {len(greeks_no_hedge)}")

# 7. Delta neto cercano a 0 con hedge
if hasattr(results_with_hedge, 'hedge_trades') and results_with_hedge.hedge_trades:
    mean_delta_neto = abs(delta_neto.mean())
    print(f"\n7. Delta neto medio (con hedge): {mean_delta_neto:.4f}")
    print(f"   Cercano a 0: {'OK' if mean_delta_neto < 0.1 else 'REVISAR'}")

print("\n" + "=" * 60)
print("Verificaciones completadas.")

VERIFICACIONES FINALES
1. Trades sin hedge: 60, con hedge: 60
   Consistente: OK

2. Trades abiertos: sin hedge=0, con hedge=0
   OK (puede haber 0-1 al final)

3. Strikes son enteros (ATM): OK

4. Entry prices razonables ($3-$60): OK
   Min: $8.60, Max: $32.74

5. Hedge trades registrados: OK
   Total rebalanceos: 779

6. Griegas calculadas: OK
   Dias con griegas: 1258

7. Delta neto medio (con hedge): 0.1055
   Cercano a 0: REVISAR

Verificaciones completadas.


## 12. Analisis de Griegas por Posicion Individual

Visualizacion de la evolucion de las griegas para una posicion especifica.
Permite analizar la relacion inversa entre gamma y theta, asi como otras griegas.

In [ ]:
def calculate_position_greeks_timeseries(position, market_data):
    """
    Calcula la evolucion de las griegas para una posicion individual.

    Parameters
    ----------
    position : StraddlePosition
        Posicion a analizar
    market_data : pd.DataFrame
        Datos de mercado consolidados

    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: delta, gamma, vega, theta, rho
    """
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)

    # Filtrar fechas en el lifetime de la posición
    entry_date = pd.Timestamp(position.entry_date)
    expiry_date = pd.Timestamp(position.expiry_date)

    position_dates = market_data_copy[
        (market_data_copy.index >= entry_date) &
        (market_data_copy.index <= expiry_date)
    ].index

    greeks_data = {
        'delta': [],
        'gamma': [],
        'vega': [],
        'theta': [],
        'rho': [],
        'time_to_expiry': [],
        'spot_price': []
    }

    for date in position_dates:
        date_str = str(date.date())
        T = position.time_to_expiry(date_str)

        if T > 0:
            row = market_data_copy.loc[date]
            S = row['close_spy']
            sigma = row['close_vix']
            q = row['q_yield']

            days_remaining = position.days_to_expiry(date_str)
            curve_row = {
                30: row[30],
                90: row[90],
                180: row[180],
                365: row[365]
            }
            r = get_risk_free_rate(days_remaining, curve_row)

            greeks = calculate_straddle_greeks(S, position.strike, T, r, q, sigma)

            greeks_data['delta'].append(greeks['delta'])
            greeks_data['gamma'].append(greeks['gamma'])
            greeks_data['vega'].append(greeks['vega'])
            greeks_data['theta'].append(greeks['theta'])
            greeks_data['rho'].append(greeks['rho'])
            greeks_data['time_to_expiry'].append(T)
            greeks_data['spot_price'].append(S)
        else:
            row = market_data_copy.loc[date]
            S = row['close_spy']

            if S > position.strike:
                delta_expiry = 1.0
            elif S < position.strike:
                delta_expiry = -1.0
            else:
                delta_expiry = 0.0

            greeks_data['delta'].append(delta_expiry)
            greeks_data['gamma'].append(0.0)
            greeks_data['vega'].append(0.0)
            greeks_data['theta'].append(0.0)
            greeks_data['rho'].append(0.0)
            greeks_data['time_to_expiry'].append(0.0)
            greeks_data['spot_price'].append(S)

    return pd.DataFrame(greeks_data, index=position_dates)

In [ ]:
# Listar todas las posiciones disponibles
print("POSICIONES DISPONIBLES PARA ANALISIS")
print("=" * 100)
print(f"{'Index':>5s} {'Entry Date':>12s} {'Expiry Date':>12s} {'Strike':>8s} {'Tenor':>6s} {'Status':>8s} {'Realized P&L':>13s}")
print("-" * 100)

for idx, trade in enumerate(results_with_hedge.trades):
    pnl_str = f"${trade.realized_pnl:.2f}" if trade.realized_pnl is not None else "N/A"
    print(f"{idx:>5d} {trade.entry_date:>12s} {trade.expiry_date:>12s} {trade.strike:>8.0f} {trade.tenor_days:>6d} {trade.status:>8s} {pnl_str:>13s}")

print(f"\nTotal posiciones: {len(results_with_hedge.trades)}")

POSICIONES DISPONIBLES PARA ANALISIS
Index   Entry Date  Expiry Date   Strike  Tenor   Status  Realized P&L
----------------------------------------------------------------------------------------------------
    0   2019-01-02   2019-02-01      250     30   closed         $6.80
    1   2019-02-01   2019-03-04      270     30   closed        $-0.84
    2   2019-03-01   2019-04-01      280     30   closed        $-2.63
    3   2019-04-01   2019-05-01      286     30   closed        $-2.10
    4   2019-05-01   2019-05-31      292     30   closed        $-1.92
    5   2019-06-03   2019-07-03      275     30   closed        $11.96
    6   2019-07-01   2019-07-31      296     30   closed        $-2.57
    7   2019-08-01   2019-09-03      295     30   closed        $-1.27
    8   2019-09-03   2019-10-03      291     30   closed        $-2.03
    9   2019-10-01   2019-10-31      293     30   closed        $-4.53
   10   2019-11-01   2019-12-02      306     30   closed        $-0.78
   11   20

In [ ]:
# ===================================================================
# SELECCIONAR POSICION A ANALIZAR (cambiar el indice)
# ===================================================================
POSITION_INDEX = 5  # Cambiar este numero para analizar diferentes posiciones

selected_position = results_with_hedge.trades[POSITION_INDEX]

print(f"POSICION SELECCIONADA: #{POSITION_INDEX}")
print("=" * 60)
print(f"Entry Date:      {selected_position.entry_date}")
print(f"Expiry Date:     {selected_position.expiry_date}")
print(f"Strike:          ${selected_position.strike:.2f}")
print(f"Tenor (days):    {selected_position.tenor_days}")
print(f"Entry Price:     ${selected_position.entry_price:.2f}")
print(f"Status:          {selected_position.status}")
if selected_position.realized_pnl is not None:
    print(f"Realized P&L:    ${selected_position.realized_pnl:.2f}")
print()

print("Calculando griegas de la posicion...")
position_greeks = calculate_position_greeks_timeseries(selected_position, market_data)
print(f"Griegas calculadas para {len(position_greeks)} dias.")
print()

print("ESTADISTICAS DE GRIEGAS - POSICION INDIVIDUAL")
print("=" * 60)
print(position_greeks[['delta', 'gamma', 'vega', 'theta']].describe())

POSICION SELECCIONADA: #5
Entry Date:      2019-06-03
Expiry Date:     2019-07-03
Strike:          $275.00
Tenor (days):    30
Entry Price:     $11.84
Status:          closed
Realized P&L:    $11.96

Calculando griegas de la posicion...
Griegas calculadas para 23 dias.

ESTADISTICAS DE GRIEGAS - POSICION INDIVIDUAL
           delta      gamma       vega      theta
count  23.000000  23.000000  23.000000  23.000000
mean    0.826112   0.019826   0.170461  -0.060907
std     0.252993   0.018967   0.201298   0.059061
min     0.003963   0.000000   0.000000  -0.196360
25%     0.794438   0.002728   0.010802  -0.093427
50%     0.956482   0.011454   0.060969  -0.035982
75%     0.992559   0.031618   0.253845  -0.010524
max     1.000000   0.053703   0.627077   0.000000


In [ ]:
# Grafico principal: Gamma vs Theta (relacion inversa)
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Gamma en eje izquierdo
fig.add_trace(
    go.Scatter(
        x=position_greeks.index,
        y=position_greeks['gamma'],
        mode='lines',
        name='Gamma',
        line=dict(color='green', width=2)
    ),
    secondary_y=False
)

# Theta en eje derecho
fig.add_trace(
    go.Scatter(
        x=position_greeks.index,
        y=position_greeks['theta'],
        mode='lines',
        name='Theta',
        line=dict(color='red', width=2)
    ),
    secondary_y=True
)

fig.add_hline(y=0, line_dash='dash', line_color='gray', secondary_y=False)
fig.add_hline(y=0, line_dash='dash', line_color='gray', secondary_y=True)

fig.update_yaxes(title_text="Gamma", secondary_y=False)
fig.update_yaxes(title_text="Theta (diario)", secondary_y=True)

fig.update_layout(
    title=f'Evolucion de Gamma vs Theta - Posicion #{POSITION_INDEX} (Strike ${selected_position.strike:.0f})',
    xaxis_title='Fecha',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [ ]:
# Grafico: Todas las griegas en subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Delta', 'Gamma', 'Vega', 'Theta'),
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

fig.add_trace(
    go.Scatter(x=position_greeks.index, y=position_greeks['delta'],
               mode='lines', name='Delta', line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=position_greeks.index, y=position_greeks['gamma'],
               mode='lines', name='Gamma', line=dict(color='green')),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=position_greeks.index, y=position_greeks['vega'],
               mode='lines', name='Vega', line=dict(color='orange')),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=position_greeks.index, y=position_greeks['theta'],
               mode='lines', name='Theta', line=dict(color='red')),
    row=2, col=2
)

for row in [1, 2]:
    for col in [1, 2]:
        fig.add_hline(y=0, line_dash='dot', line_color='gray', row=row, col=col)

fig.update_layout(
    title=f'Evolucion de Todas las Griegas - Posicion #{POSITION_INDEX}',
    height=600,
    showlegend=False
)

fig.show()